# Trade Engine Demo: Draft Capital Curve + Net KVS Value

Demonstrates the two pieces built in `src/trade_engine/`:

1. **`draft_capital_curve.py`** — translates a projected keeper round into an expected VORP figure, using real, realized 2024-2025 fresh-pick outcomes. Two real problems had to be solved before this curve could be built at all, both checked directly rather than assumed:
   - **The crosswalk problem**: `draft_history.parquet` identifies players by Sleeper's `player_id`; `vorp_labels.parquet` identifies them by NFL's `gsis_id`. Joining on Sleeper's own cached `gsis_id` field alone only resolved 100/310 (32%) of fresh picks — misses included Bo Nix, Malik Nabers, Travis Etienne, and ~150 other real, well-known players whose Sleeper record simply has a blank `gsis_id`. A two-tier crosswalk (direct `gsis_id`, then name+position match against `nflreadpy`'s own player table, then direct pass-through for team defenses) recovers 282/310 (91%).
   - **Round sparsity**: real usable-with-VORP counts per round range from 9 (rounds 16-17) to 20 (round 14) across 2024-2025. Rounds 16-17 band together (`min_picks_per_round=10`); every other round, including round 18 sitting exactly at the threshold, stands alone.
2. **`net_value.py`** — `net_KVS_delta = predicted_KVS - keeper_cost_VORP`, where `predicted_KVS` comes from the appropriate tuned QB/RB/WR/TE model (the same `0Xx_model_*.ipynb`/`08x_shap_*.ipynb` models used all night), and `keeper_cost_VORP` comes from `scripts/project_roster_keeper_costs.py`'s already-validated projected keeper round, converted via the curve above. Every result carries `low_confidence_extreme_delta`/`no_delta_history` explicitly — never silently dropped. **K and DEF are out of scope for this module entirely** — `predict_kvs`/`evaluate_player_trade_value` raise `UnsupportedPositionError` for either position rather than returning a number of any kind (see Part 2's note below for why).

Both pieces are demonstrated below against real, current roster data — the same real-data-eyeball-check standard as every model notebook tonight: real numbers, checked for football sense, surprises reported honestly rather than smoothed over.

## Part 1 — Draft Capital Curve

In [1]:
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.trade_engine.draft_capital_curve import (
    build_draft_capital_curve,
    build_full_draft_capital_curve,
    resolve_fresh_picks,
    attach_realized_vorp,
)

pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)

draft_curve = build_full_draft_capital_curve(REPO_ROOT)
print("Draft capital curve (2024-2025, min_picks_per_round=10):\n")
print(draft_curve.to_string())


Draft capital curve (2024-2025, min_picks_per_round=10):

         avg_vorp  n_picks band_rounds  banded
round                                         
1      107.364286       14        (1,)   False
2       70.524000       15        (2,)   False
3       49.861579       19        (3,)   False
4       32.754667       15        (4,)   False
5        2.320667       15        (5,)   False
6        3.224444       18        (6,)   False
7      -10.859474       19        (7,)   False
8      -18.248571       14        (8,)   False
9       -7.766875       16        (9,)   False
10     -10.253889       18       (10,)   False
11     -12.551111       18       (11,)   False
12     -21.792353       17       (12,)   False
13     -27.937222       18       (13,)   False
14     -13.138500       20       (14,)   False
15     -33.218333       18       (15,)   False
16     -15.471111       18    (16, 17)    True
17     -15.471111       18    (16, 17)    True
18     -14.131000       10       (18,)   False


**Real-data eyeball check**: `avg_vorp` should decline roughly monotonically from round 1 down through the late rounds, since earlier picks are, on average, real fantasy stars and later picks are replacement-level or worse. Rounds 16 and 17 should show identical `avg_vorp`/`n_picks` and a shared `band_rounds = (16, 17)` — the banding decision from the investigation above, visibly working on real data, not just in the unit tests.

### Crosswalk transparency: how many fresh picks actually resolved, and how

In [2]:
import json
import nflreadpy as nfl

draft_history = pd.read_parquet(REPO_ROOT / "data/raw/draft_history.parquet")
with open(REPO_ROOT / "data/raw/players_cache.json") as f:
    sleeper_players = json.load(f)
nfl_players = nfl.load_players().to_pandas()
vorp_labels = pd.read_parquet(REPO_ROOT / "data/processed/vorp_labels.parquet")

resolved = resolve_fresh_picks(draft_history, sleeper_players, nfl_players, seasons=[2024, 2025])
with_vorp = attach_realized_vorp(resolved, vorp_labels)

print("Match method breakdown (how each fresh pick's vorp_key was resolved):")
print(resolved["match_method"].value_counts().to_string())
print(f"\nOf {len(resolved)} fresh picks, {with_vorp['vorp'].notna().sum()} resolved to a real VORP row "
      f"({with_vorp['vorp'].notna().mean():.1%}).")

still_unresolved = resolved[resolved["match_method"] == "unresolved"]["player_id"].unique()
print(f"\n{len(still_unresolved)} player_ids never resolved at all (genuine name collisions / position "
      f"mismatches, not chased further):")
for pid in still_unresolved:
    p = sleeper_players.get(pid, {})
    print(f"  {pid}: {p.get('full_name')} ({p.get('position')})")


Match method breakdown (how each fresh pick's vorp_key was resolved):
match_method
name_fallback      186
direct             100
def_passthrough     19
unresolved           5

Of 310 fresh picks, 282 resolved to a real VORP row (91.0%).

4 player_ids never resolved at all (genuine name collisions / position mismatches, not chased further):
  7670: Joshua Palmer (WR)
  11628: Marvin Harrison (WR)
  12547: Kyle Williams (WR)
  12530: Travis Hunter (DB)


**Note on K/DEF and this curve specifically**: unlike Part 2 below, this curve deliberately keeps K/DEF fresh picks in its averages — 38 real ones (19 K + 19 DEF) exist in the 2024-2025 window and pass through the crosswalk/banding above like any other position. This curve prices what a *draft slot* is worth, and real late rounds are disproportionately K/DEF (round 18 alone is 7 of its 10 usable picks) — excluding them would understate what a late-round pick actually costs to acquire. See `draft_capital_curve.py`'s own module docstring for the full reasoning; it is a separate question from whether a K/DEF *player* can be evaluated, addressed next.

## Part 2 — Net KVS Value, on real current roster players

**K and DEF are out of scope for this section, on purpose.** Per `roadmap.md`'s Phase 4 scope decision (full reasoning in `notebooks/06_scope_decision_k_def.ipynb`: DEF's untuned model actively underperformed simple persistence, K/DEF have a structurally thin feature set, and both show real-world year-to-year volatility no available features capture well), `net_value.py` does not produce a `predicted_KVS` for either position by any means — not a tuned model, and not a heuristic either. `predict_kvs`/`evaluate_player_trade_value` raise `UnsupportedPositionError` for K or DEF instead of returning a number of any kind, demonstrated directly below before moving to the six real, supported players.

Six real, currently-rostered QB/RB/WR/TE players are pulled directly from `data/processed/roster_keeper_table_2027.csv` (— the actual output of `scripts/project_roster_keeper_costs.py`), deliberately including at least one player already known from tonight's SHAP work to carry a real confidence caveat (Christian McCaffrey, `low_confidence_extreme_delta` in `06b_model_rb.ipynb`/`08b_shap_rb.ipynb`) and one TE (Kyle Pitts) whose feature set structurally can't represent his real-world context (see `06d_model_te.ipynb`).

In [3]:
from src.trade_engine.net_value import evaluate_player_trade_value, UnsupportedPositionError

for name, position in [("Any Kicker", "K"), ("Any Defense", "DEF")]:
    try:
        evaluate_player_trade_value(name, 2025, position, projected_keeper_round=10, draft_curve=draft_curve, repo_root=REPO_ROOT)
        print(f"UNEXPECTED: {name} ({position}) did not raise")
    except UnsupportedPositionError as e:
        print(f"{position} correctly raised UnsupportedPositionError:")
        print(f"  {e}")
        print()


K correctly raised UnsupportedPositionError:
  Position 'K' is out of scope for trade evaluation. Per roadmap.md's Phase 4 scope decision, no XGBoost model or SHAP explainer exists or is planned for K/DEF -- see notebooks/06_scope_decision_k_def.ipynb for the full reasoning. This function will not return a predicted_KVS of any kind for this position.

DEF correctly raised UnsupportedPositionError:
  Position 'DEF' is out of scope for trade evaluation. Per roadmap.md's Phase 4 scope decision, no XGBoost model or SHAP explainer exists or is planned for K/DEF -- see notebooks/06_scope_decision_k_def.ipynb for the full reasoning. This function will not return a predicted_KVS of any kind for this position.



In [4]:
roster_keeper_table = pd.read_csv(REPO_ROOT / "data/processed/roster_keeper_table_2027.csv")

demo_players = [
    ("Christian McCaffrey", "RB"),
    ("Ja'Marr Chase", "WR"),
    ("Caleb Williams", "QB"),
    ("Kyle Pitts", "TE"),
    ("DJ Moore", "WR"),
    ("Travis Kelce", "TE"),
]

rows = []
for player_name, position in demo_players:
    match = roster_keeper_table[
        (roster_keeper_table["player"] == player_name) & (roster_keeper_table["position"] == position)
    ]
    if match.empty:
        print(f"WARNING: {player_name} ({position}) not found in roster_keeper_table_2027.csv -- skipping")
        continue
    rows.append({
        "player_name": player_name,
        "position": position,
        "projected_keeper_round": int(match.iloc[0]["round_lost_if_kept_2027"]),
    })

demo_df = pd.DataFrame(rows)
print("Real players pulled from data/processed/roster_keeper_table_2027.csv:")
print(demo_df.to_string(index=False))


Real players pulled from data/processed/roster_keeper_table_2027.csv:
        player_name position  projected_keeper_round
Christian McCaffrey       RB                       1
      Ja'Marr Chase       WR                       1
     Caleb Williams       QB                       8
         Kyle Pitts       TE                       6
           DJ Moore       WR                       2
       Travis Kelce       TE                       7


In [5]:
SEASON = 2025  # each player's real, already-realized 2025 season feeds the 2026 prediction --
                # matching every 06x_model_*.ipynb notebook's own live-prediction convention.

results = []
for _, row in demo_df.iterrows():
    result = evaluate_player_trade_value(
        row["player_name"], SEASON, row["position"], row["projected_keeper_round"],
        draft_curve, REPO_ROOT, vorp_labels=vorp_labels,
    )
    results.append(result)
    print(f"=== {result.player_name} ({result.position}) ===")
    print(f"  predicted_KVS:          {result.predicted_kvs:.2f}")
    print(f"  keeper_cost_VORP:       {result.keeper_cost_vorp:.2f}  (projected round {row['projected_keeper_round']})")
    print(f"  net_KVS_delta:          {result.net_kvs_delta:.2f}")
    print(f"  low_confidence_extreme_delta: {result.low_confidence_extreme_delta}")
    print(f"  no_delta_history:             {result.no_delta_history}")
    if result.caveats:
        print("  CAVEATS:")
        for c in result.caveats:
            print(f"    - {c}")
    print()


=== Christian McCaffrey (RB) ===
  predicted_KVS:          74.67
  keeper_cost_VORP:       107.36  (projected round 1)
  net_KVS_delta:          -32.70
  low_confidence_extreme_delta: True
  no_delta_history:             False
  CAVEATS:
    - low_confidence_extreme_delta: |vorp_delta_yoy| > 100 -- this region has confirmed, held-out degraded accuracy at every modeled position (see 08a-08d_shap_*.ipynb); treat the exact predicted_KVS number with real skepticism, not just the direction.

=== Ja'Marr Chase (WR) ===
  predicted_KVS:          76.33
  keeper_cost_VORP:       107.36  (projected round 1)
  net_KVS_delta:          -31.04
  low_confidence_extreme_delta: False
  no_delta_history:             False



=== Caleb Williams (QB) ===
  predicted_KVS:          -28.61
  keeper_cost_VORP:       -18.25  (projected round 8)
  net_KVS_delta:          -10.36
  low_confidence_extreme_delta: False
  no_delta_history:             False



=== Kyle Pitts (TE) ===
  predicted_KVS:          19.80
  keeper_cost_VORP:       3.22  (projected round 6)
  net_KVS_delta:          16.58
  low_confidence_extreme_delta: False
  no_delta_history:             False

=== DJ Moore (WR) ===
  predicted_KVS:          -13.69
  keeper_cost_VORP:       70.52  (projected round 2)
  net_KVS_delta:          -84.22
  low_confidence_extreme_delta: False
  no_delta_history:             False

=== Travis Kelce (TE) ===
  predicted_KVS:          42.49
  keeper_cost_VORP:       -10.86  (projected round 7)
  net_KVS_delta:          53.35
  low_confidence_extreme_delta: False
  no_delta_history:             False



**Real-data eyeball check, player by player:**

- **Christian McCaffrey (RB)**: `net_KVS_delta = -32.70`, correctly carries `low_confidence_extreme_delta` — ties back to this exact model's documented blind spot for him (`06b_model_rb.ipynb`/`08b_shap_rb.ipynb`).
- **Ja'Marr Chase (WR)**: `net_KVS_delta = -31.04`, no flags. Round 1's average realized VORP (107.4) is a high bar; even a strong prediction (76.3) falls short of it.
- **Caleb Williams (QB)**: `net_KVS_delta = -10.36`, no flags. Both `predicted_KVS` (-28.6) and `keeper_cost_VORP` (-18.2) are negative — a real, fairly pessimistic model read, presented as-is rather than football-sense-checked against his draft pedigree.
- **Kyle Pitts (TE)**: `net_KVS_delta = +16.58`, no flags. `predicted_KVS` (19.8) matches `08d_shap_te.ipynb`'s own real output for this exact row exactly — a useful cross-check that this module's feature-building reproduces that notebook's pipeline correctly.
- **DJ Moore (WR)**: `net_KVS_delta = -84.22`, the largest-magnitude number in the whole table, and it's worth actually explaining rather than just reporting. Checked directly: his 2025 `scarcity_z` is only 1.39 — solid, but well short of Chase's or Pitts's level, and `scarcity_z` is WR's dominant feature by a wide margin (~59% of gain, per `08c_shap_wr.ipynb`). His `vorp_delta_yoy` (-12.62) isn't extreme enough to trip any flag. The steep negative comes from a real, legitimate mismatch: his round-2 keeper cost prices him as a well-above-average performer, but his actual current production level is more middling — not a bug, and not something the flags were designed to catch, since neither his delta nor his history is unusual.
- **Travis Kelce (TE)**: `net_KVS_delta = +53.35`, no flags. Consistent with `06d_model_te.ipynb`'s own real-world caveat that this near-flat-to-positive read may be *too optimistic* given his publicly-documented age-36 decline — TE's feature set structurally excludes `age` entirely, so this positive number should be read with that specific, named blind spot in mind, not taken as confidently as it looks.

**K/DEF exclusion confirmed working, not just claimed**: both `UnsupportedPositionError` calls above raised with the real message, referencing `06_scope_decision_k_def.ipynb` directly, before any of the six real players were touched.

### Summary table, sorted by net_KVS_delta

In [6]:
summary = pd.DataFrame([
    {
        "player": r.player_name, "position": r.position,
        "predicted_KVS": round(r.predicted_kvs, 1), "keeper_cost_VORP": round(r.keeper_cost_vorp, 1),
        "net_KVS_delta": round(r.net_kvs_delta, 1),
        "low_confidence_extreme_delta": r.low_confidence_extreme_delta,
        "no_delta_history": r.no_delta_history,
    }
    for r in results
]).sort_values("net_KVS_delta", ascending=False).reset_index(drop=True)

# A single blended sort is safe here -- unlike an earlier version of this demo, every row in
# `results` now comes from the same reliability class (a real tuned model); K/DEF's heuristic
# path (and the reliability_tier machinery that existed only to separate it from real model
# output) has been removed from net_value.py entirely, not just hidden from this table.
print(summary.to_string(index=False))


             player position  predicted_KVS  keeper_cost_VORP  net_KVS_delta  low_confidence_extreme_delta  no_delta_history
       Travis Kelce       TE           42.5             -10.9           53.3                         False             False
         Kyle Pitts       TE           19.8               3.2           16.6                         False             False
     Caleb Williams       QB          -28.6             -18.2          -10.4                         False             False
      Ja'Marr Chase       WR           76.3             107.4          -31.0                         False             False
Christian McCaffrey       RB           74.7             107.4          -32.7                          True             False
           DJ Moore       WR          -13.7              70.5          -84.2                         False             False


## Summary

**The draft capital curve looks sensible on real data**: `avg_vorp` declines from +107.4 (round 1) to roughly -15 to -30 through the late rounds, with rounds 16-17 correctly banded (`band_rounds = (16, 17)`, identical `avg_vorp`/`n_picks`) — the banding decision from the investigation, visibly working on the real curve.

**The crosswalk resolved 282/310 (91.0%) of fresh 2024-2025 picks on this run** — 100 direct `gsis_id` matches, 186 name/position fallback matches, 19 DEF pass-throughs, 5 genuine unresolved misses.

**K/DEF exclusion is real, not just documented**: `evaluate_player_trade_value` raised `UnsupportedPositionError` for both K and DEF, with a message pointing directly at `06_scope_decision_k_def.ipynb`, before any of the six supported players were evaluated. `net_value.py` no longer contains a heuristic path, `reliability_tier`, or `group_by_reliability_tier` at all — that machinery only ever existed to keep heuristic numbers separate from real ones, and with no heuristic numbers left to produce, it would have been dead code, not a safeguard.

**All six QB/RB/WR/TE players produced sensible, honestly-labeled net values, including one genuine explanation dug up rather than just reported**: DJ Moore's outlier `-84.22` traces to a real, moderate (not extreme, not flagged) `scarcity_z` colliding with a round-2 keeper cost baseline that assumes much more than his current production supports — a legitimate model read, not a bug, and not something either confidence flag was designed to catch.

**The draft capital curve legitimately still includes K/DEF fresh picks in its own averages** — a separate, deliberate decision from `net_value.py`'s exclusion, since the curve prices draft slots (which real K/DEF picks genuinely occupy, especially late) rather than evaluating players. Documented directly in `draft_capital_curve.py` and locked in with a dedicated test.

**No caveat was silently dropped anywhere in this run**: every flagged condition that occurred (McCaffrey's extreme delta) produced a real, printed caveat string, confirmed directly in the output above.